# Timestep-Flexible S5 Reconstruction

Minimal Colab notebook for training the timestep-flexible Brain2Text24 S5 decoder and plotting `20 ms` / `40 ms` validation diagnostics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl')
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only || true

%cd {REPO_DIR}
!pip install -q torch pandas matplotlib

RAW_CACHE_ROOT = Path('/content/drive/MyDrive/utah_ssl/data/cache_v1')
SMOOTHED_CACHE_ROOT = Path('/content/drive/MyDrive/utah_ssl/data/cache_v1_smoothed_sigma2p0')
TRIM_SCRIPT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments' / 'trim_area6v_cache.py'
OUTPUT_ROOT = Path('/content/drive/MyDrive/utah_ssl/outputs/timestep_flexible_ssm')
EXPERIMENTS_DIR = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
PYTHONPATH = str(EXPERIMENTS_DIR)
os.environ['PYTHONPATH'] = PYTHONPATH
if PYTHONPATH not in sys.path:
    sys.path.insert(0, PYTHONPATH)

for cache_root in (RAW_CACHE_ROOT, SMOOTHED_CACHE_ROOT):
    dataset_root = cache_root / 'brain2text24'
    if dataset_root.exists():
        subprocess.run(
            ['python', str(TRIM_SCRIPT), '--cache-root', str(cache_root)],
            check=True,
            cwd=str(REPO_DIR),
        )

print('RAW_CACHE_ROOT:', RAW_CACHE_ROOT)
print('SMOOTHED_CACHE_ROOT:', SMOOTHED_CACHE_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
from pathlib import Path

RUN_NAME = 'timestep_flexible_s5_tx_only_colab'
DATASET = 'brain2text24'
FEATURE_MODE = 'tx_only'
CACHE_ROOT = RAW_CACHE_ROOT
MAX_STEPS = 12000
BATCH_SIZE = 64
LEARNING_RATE = 1e-2
MIN_LEARNING_RATE = 1e-4
ADAM_EPSILON = 1e-1
SESSION_ADAPTER = True
NORMALIZATION_MODE = 'global'
TRAIN_BIN_SIZE_MS = 20
EVAL_BIN_SIZES_MS = (20, 40)
PATCH_SIZE_MS = 280
PATCH_STRIDE_MS = 80
RESUME_LATEST = False

RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RUN_DIR = OUTPUT_ROOT / RUN_NAME
progress_path = RUN_DIR / 'progress.jsonl'
summary_path = RUN_DIR / 'summary.json'

if not progress_path.exists():
    print(f'No progress log yet at: {progress_path}')
    print('Run the training cell first, then come back here to preview the curves.')
else:
    progress = [json.loads(line) for line in progress_path.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(progress)

    train_df = df[df.get('event').eq('timestep_flexible_train_report')] if 'event' in df.columns else pd.DataFrame()
    val_df = df[df.get('event').eq('timestep_flexible_val_report')] if 'event' in df.columns else pd.DataFrame()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    if not train_df.empty and 'step' in train_df and 'train_ctc_bpphone' in train_df:
        axes[0].plot(train_df['step'], train_df['train_ctc_bpphone'], label='train CTC')
    if not val_df.empty:
        if 'val_20ms_ctc_bpphone' in val_df:
            axes[0].plot(val_df['step'], val_df['val_20ms_ctc_bpphone'], marker='o', label='val 20 ms CTC')
        if 'val_40ms_ctc_bpphone' in val_df:
            axes[0].plot(val_df['step'], val_df['val_40ms_ctc_bpphone'], marker='o', label='val 40 ms CTC')
    axes[0].set_title('Timestep-Flexible S5 CTC')
    axes[0].set_xlabel('step')
    axes[0].set_ylabel('bits / phoneme')
    axes[0].legend()

    if not val_df.empty:
        if 'val_20ms_phoneme_error_rate' in val_df:
            axes[1].plot(val_df['step'], val_df['val_20ms_phoneme_error_rate'], marker='o', label='val 20 ms PER')
        if 'val_40ms_phoneme_error_rate' in val_df:
            axes[1].plot(val_df['step'], val_df['val_40ms_phoneme_error_rate'], marker='o', label='val 40 ms PER')
    axes[1].set_title('Timestep-Flexible S5 PER')
    axes[1].set_xlabel('step')
    axes[1].set_ylabel('PER')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print(json.dumps(summary.get('metrics', {}), indent=2))


## Recompute Split Stats

Run this cell when `FEATURE_MODE` changes or after trimming/rebuilding the cache. It refreshes the canonical `20 ms` competition-train global normalization stats used by the timestep-flexible trainer. The trainer derives rebinned `40 ms` stats from the train split at runtime.

In [ ]:
import json
import subprocess

from recompute_split_feature_stats import resolve_precomputed_split_stats_path

RECOMPUTE_SPLIT_STATS = True
SPLIT_STATS_SCRIPT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments' / 'recompute_split_feature_stats.py'

stats_output_path = resolve_precomputed_split_stats_path(
    cache_root=CACHE_ROOT,
    dataset=DATASET,
    train_split_name='competition_train',
    feature_mode=FEATURE_MODE,
    preferred_path=None,
)

trim_cmd = ['python', str(TRIM_SCRIPT), '--cache-root', str(CACHE_ROOT)]
print('Running:', ' '.join(trim_cmd))
trim_result = subprocess.run(trim_cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
print('trim returncode:', trim_result.returncode)
if trim_result.stdout:
    print('\nTRIM STDOUT\n')
    print(trim_result.stdout)
if trim_result.stderr:
    print('\nTRIM STDERR\n')
    print(trim_result.stderr)
if trim_result.returncode != 0:
    raise RuntimeError('Area-6v cache trim failed before stats recompute.')

if RECOMPUTE_SPLIT_STATS:
    cmd = [
        'python', str(SPLIT_STATS_SCRIPT),
        '--cache-root', str(CACHE_ROOT),
        '--dataset', DATASET,
        '--feature-mode', FEATURE_MODE,
        '--output-path', str(stats_output_path),
        '--overwrite',
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
    print('stats returncode:', result.returncode)
    if result.stdout:
        print('\nSTATS STDOUT\n')
        print(result.stdout)
    if result.stderr:
        print('\nSTATS STDERR\n')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Split-stat recompute failed.')

print('stats_output_path:', stats_output_path)


In [ ]:
import json
import os
import shlex
import subprocess
import threading
import time

PACKAGE_ROOT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'

cmd = [
    'python', '-m', 'timestep_flexible_ssm.train',
    '--cache-root', str(CACHE_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', RUN_NAME,
    '--dataset', DATASET,
    '--feature-mode', FEATURE_MODE,
    '--normalization-mode', NORMALIZATION_MODE,
    '--train-bin-size-ms', str(TRAIN_BIN_SIZE_MS),
    '--eval-bin-sizes-ms', ','.join(str(item) for item in EVAL_BIN_SIZES_MS),
    '--patch-size-ms', str(PATCH_SIZE_MS),
    '--patch-stride-ms', str(PATCH_STRIDE_MS),
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--min-learning-rate', str(MIN_LEARNING_RATE),
    '--adam-epsilon', str(ADAM_EPSILON),
    '--val-every-steps', '100',
    '--checkpoint-every-steps', '500',
    '--progress-every-steps', '25',
]
if RESUME_LATEST:
    cmd.append('--resume-latest')
if not SESSION_ADAPTER:
    cmd.append('--disable-session-adapter')
full_cmd = ' '.join(shlex.quote(part) for part in cmd)
pythonpath = f"{PACKAGE_ROOT}:{os.environ.get('PYTHONPATH', '')}"
env = dict(os.environ)
env['PYTHONPATH'] = pythonpath
progress_path = RUN_DIR / 'progress.jsonl'

print(full_cmd)

def _tail_progress(stop_event):
    seen_lines = 0
    while not stop_event.is_set():
        if progress_path.exists():
            lines = [line for line in progress_path.read_text().splitlines() if line.strip()]
            new_lines = lines[seen_lines:]
            for line in new_lines:
                try:
                    payload = json.loads(line)
                except json.JSONDecodeError:
                    print(line)
                    continue
                event = payload.get('event', 'progress')
                step = payload.get('step', '?')
                if event == 'timestep_flexible_train_report':
                    print(f"[train] step={step} ctc={payload.get('train_ctc_bpphone'):.4f} lr={payload.get('learning_rate'):.6f} elapsed={payload.get('elapsed_seconds'):.1f}s")
                elif event == 'timestep_flexible_val_report':
                    msg = [f"[val] step={step}"]
                    if 'val_20ms_ctc_bpphone' in payload:
                        msg.append(f"20ms_ctc={payload['val_20ms_ctc_bpphone']:.4f}")
                    if 'val_20ms_phoneme_error_rate' in payload:
                        msg.append(f"20ms_per={payload['val_20ms_phoneme_error_rate']:.4f}")
                    if 'val_40ms_ctc_bpphone' in payload:
                        msg.append(f"40ms_ctc={payload['val_40ms_ctc_bpphone']:.4f}")
                    if 'val_40ms_phoneme_error_rate' in payload:
                        msg.append(f"40ms_per={payload['val_40ms_phoneme_error_rate']:.4f}")
                    print(' '.join(msg))
                else:
                    print(payload)
            seen_lines = len(lines)
        time.sleep(5)

stop_event = threading.Event()
tail_thread = threading.Thread(target=_tail_progress, args=(stop_event,), daemon=True)
tail_thread.start()

process = subprocess.Popen(
    cmd,
    cwd=str(PACKAGE_ROOT),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
try:
    for line in process.stdout:
        if line.strip():
            print(line.rstrip())
    return_code = process.wait()
finally:
    stop_event.set()
    tail_thread.join(timeout=1)

if progress_path.exists():
    final_lines = [line for line in progress_path.read_text().splitlines() if line.strip()]
    for line in final_lines[-5:]:
        try:
            print(json.loads(line))
        except json.JSONDecodeError:
            print(line)

if return_code != 0:
    raise RuntimeError(f'Training command failed with exit code {return_code}')


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

summary_path = RUN_DIR / 'summary.json'
progress_path = RUN_DIR / 'progress.jsonl'
summary = json.loads(summary_path.read_text())
progress = [json.loads(line) for line in progress_path.read_text().splitlines() if line.strip()]
progress_df = pd.DataFrame(progress)
train_df = progress_df[progress_df['event'] == 'timestep_flexible_train_report'].copy()
val_df = progress_df[progress_df['event'] == 'timestep_flexible_val_report'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
if not train_df.empty:
    axes[0].plot(train_df['step'], train_df['train_ctc_bpphone'], label='train CTC')
if not val_df.empty:
    if 'val_20ms_ctc_bpphone' in val_df:
        axes[0].plot(val_df['step'], val_df['val_20ms_ctc_bpphone'], marker='o', label='val 20 ms CTC')
    if 'val_40ms_ctc_bpphone' in val_df:
        axes[0].plot(val_df['step'], val_df['val_40ms_ctc_bpphone'], marker='o', label='val 40 ms CTC')
    if 'val_20ms_phoneme_error_rate' in val_df:
        axes[1].plot(val_df['step'], val_df['val_20ms_phoneme_error_rate'], marker='o', label='val 20 ms PER')
    if 'val_40ms_phoneme_error_rate' in val_df:
        axes[1].plot(val_df['step'], val_df['val_40ms_phoneme_error_rate'], marker='o', label='val 40 ms PER')
axes[0].set_title('Timestep-Flexible S5 CTC')
axes[0].set_xlabel('step')
axes[0].set_ylabel('bits / phoneme')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].set_title('Timestep-Flexible S5 PER')
axes[1].set_xlabel('step')
axes[1].set_ylabel('PER')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

print('best step:', summary.get('best_step'))
print('final metrics:', json.dumps(summary.get('metrics', {}), indent=2))
print('best metrics:', json.dumps(summary.get('best_metrics', {}), indent=2))
if not val_df.empty:
    columns = ['step']
    for name in [
        'val_20ms_ctc_bpphone',
        'val_40ms_ctc_bpphone',
        'val_20ms_phoneme_error_rate',
        'val_40ms_phoneme_error_rate',
    ]:
        if name in val_df.columns:
            columns.append(name)
    display(val_df[columns].tail(10))


## Willett GRU Baseline Check

Validation-only comparison for the saved `willett_tx_only_area6v_colab` GRU checkpoint. This keeps patch duration / hop fixed in milliseconds and resamples `40 ms` patch tokens back to the checkpoint's original `14`-bin token width before the GRU forward pass.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader

if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

from recompute_split_feature_stats import (
    load_precomputed_split_feature_stats,
    resolve_precomputed_split_stats_path,
)
from ssl_core.ctc import compute_ctc_loss_sum, ctc_bits_per_target, ctc_greedy_decode, edit_counts
from ssl_core.patching import patch_starts
from timestep_flexible_ssm.data import (
    RebinnedSequenceDataset,
    TimestepFlexibleInputTransformConfig,
    build_timestep_flexible_problem,
    compute_rebinned_normalization_stats,
    loader_kwargs,
    make_length_aware_batch_sampler,
    prepare_timestep_flexible_inputs,
    resolve_patch_bins,
)
from willett_reconstruction.data import adapter_keys_from_rows, normalization_stats_missing_rows
from willett_reconstruction.model import WillettPhonemeModel

GRU_RUN_NAME = 'willett_tx_only_area6v_colab'
GRU_RUN_DIR = Path('/content/drive/MyDrive/utah_ssl/outputs/willett_reconstruction') / GRU_RUN_NAME
GRU_CHECKPOINT_PATH = GRU_RUN_DIR / 'checkpoint_best.pt'
GRU_SAVED_BEST_20MS_CTC = 1.8476455153996616
GRU_SAVED_BEST_20MS_PER = 0.3724146472360979
S5_SUMMARY_PATH = RUN_DIR / 'summary.json'

if not GRU_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Missing GRU checkpoint: {GRU_CHECKPOINT_PATH}. '
        'Mount Drive and confirm the Willett baseline run is present before running this cell.'
    )


def _feature_dim_from_problem(problem, feature_mode):
    row = problem['train_rows'][0]
    if feature_mode == 'tx_only':
        return int(row.n_tx_features)
    return int(row.n_tx_features + row.n_sbp_features)


def _extract_s5_metric(summary_dict, bin_size_ms, metric_name):
    best_metrics = summary_dict.get('best_metrics', {})
    key = f'val_{int(bin_size_ms)}ms_{metric_name}'
    if key in best_metrics:
        return float(best_metrics[key])
    by_bin = best_metrics.get('metrics_by_bin_ms', {})
    if str(int(bin_size_ms)) in by_bin and metric_name in by_bin[str(int(bin_size_ms))]:
        return float(by_bin[str(int(bin_size_ms))][metric_name])
    raise KeyError(f'Could not find {metric_name} for {bin_size_ms} ms in summary.json')


def _load_stats_for_bin(problem, checkpoint_cfg, sample_dim, bin_size_ms):
    normalization_mode = str(checkpoint_cfg.get('normalization_mode', 'global'))
    if int(bin_size_ms) == 20 and normalization_mode == 'global':
        stats_path = resolve_precomputed_split_stats_path(
            cache_root=Path(problem['cache_root']),
            dataset=str(problem['dataset']),
            train_split_name=str(problem['train_split_name']),
            feature_mode=str(problem['feature_mode']),
            preferred_path=checkpoint_cfg.get('precomputed_split_stats_path'),
        )
        if stats_path.exists():
            (mean_t, std_t), _, loaded_path = load_precomputed_split_feature_stats(
                stats_path=stats_path,
                cache_root=Path(problem['cache_root']),
                dataset=str(problem['dataset']),
                feature_mode=str(problem['feature_mode']),
                boundary_key_mode=str(problem['boundary_key_mode']),
                train_split_name=str(problem['train_split_name']),
                val_split_name=str(problem['val_split_name']),
                expected_dim=int(sample_dim),
            )
            print(f'Loaded canonical 20 ms split stats: {loaded_path}')
            return (
                mean_t.numpy().astype(np.float32, copy=False),
                std_t.numpy().astype(np.float32, copy=False),
            )
    print(f'Computing train-derived normalization stats for {bin_size_ms} ms bins...')
    return compute_rebinned_normalization_stats(
        problem['train_rows'],
        cache_root=Path(problem['cache_root']),
        feature_mode=str(problem['feature_mode']),
        mode=normalization_mode,
        bin_size_ms=int(bin_size_ms),
    )


def _resample_patch_width(patch_2d, target_bins):
    if int(patch_2d.shape[0]) == int(target_bins):
        return patch_2d
    patch_cf = patch_2d.transpose(0, 1).unsqueeze(0)
    resized = F.interpolate(
        patch_cf,
        size=int(target_bins),
        mode='linear',
        align_corners=False,
    )
    return resized.squeeze(0).transpose(0, 1)


def _patch_resample_batch(x, input_lengths, active_patch_size_bins, active_patch_stride_bins, reference_patch_size_bins):
    token_sequences = []
    token_lengths = []
    feature_dim = int(x.shape[-1])
    patch_dim = feature_dim * int(reference_patch_size_bins)
    for sample, length_tensor in zip(x, input_lengths):
        length = max(0, int(length_tensor.item()))
        starts = patch_starts(
            length,
            patch_size=int(active_patch_size_bins),
            patch_stride=int(active_patch_stride_bins),
            policy='floor',
        )
        if not starts:
            tokens = sample.new_zeros((0, patch_dim))
        else:
            valid = sample[:length]
            patches = []
            for start in starts:
                patch = valid[start : start + int(active_patch_size_bins)]
                if int(patch.shape[0]) < int(active_patch_size_bins):
                    pad = valid.new_zeros((int(active_patch_size_bins) - int(patch.shape[0]), feature_dim))
                    patch = torch.cat([patch, pad], dim=0)
                patch = _resample_patch_width(patch, int(reference_patch_size_bins))
                patches.append(patch.reshape(-1))
            tokens = torch.stack(patches, dim=0)
        token_sequences.append(tokens)
        token_lengths.append(int(tokens.shape[0]))
    max_tokens = max(token_lengths, default=0)
    patched = x.new_zeros((int(x.shape[0]), max_tokens, patch_dim))
    for batch_idx, tokens in enumerate(token_sequences):
        if int(tokens.shape[0]) > 0:
            patched[batch_idx, : int(tokens.shape[0])] = tokens
    return patched, torch.as_tensor(token_lengths, device=input_lengths.device, dtype=torch.long)


def _forward_willett_gru_resampled(model, x, input_lengths, boundary_keys, active_bin_size_ms, patch_size_ms, patch_stride_ms, reference_patch_size_bins):
    adapted_input = model.session_input_adapter(
        x,
        boundary_keys,
        session_adapter_enabled=model.session_adapter_enabled,
    )
    active_patch_size_bins = resolve_patch_bins(
        int(patch_size_ms),
        bin_size_ms=int(active_bin_size_ms),
        field_name='patch_size_ms',
    )
    active_patch_stride_bins = resolve_patch_bins(
        int(patch_stride_ms),
        bin_size_ms=int(active_bin_size_ms),
        field_name='patch_stride_ms',
    )
    patched_inputs, token_lengths = _patch_resample_batch(
        adapted_input,
        input_lengths,
        active_patch_size_bins=int(active_patch_size_bins),
        active_patch_stride_bins=int(active_patch_stride_bins),
        reference_patch_size_bins=int(reference_patch_size_bins),
    )
    if int((token_lengths <= 0).sum().item()) > 0:
        raise ValueError('Encountered zero-length token sequence during GRU evaluation; inspect rebinned cache rows.')
    packed = nn.utils.rnn.pack_padded_sequence(
        patched_inputs,
        token_lengths.cpu(),
        batch_first=True,
        enforce_sorted=False,
    )
    initial_hidden = model._initial_hidden_state(
        batch_size=int(x.shape[0]),
        device=patched_inputs.device,
        dtype=patched_inputs.dtype,
    )
    packed_hidden, _ = model.gru(packed, initial_hidden)
    hidden, _ = nn.utils.rnn.pad_packed_sequence(
        packed_hidden,
        batch_first=True,
        total_length=patched_inputs.shape[1],
    )
    logits = model.classifier(hidden)
    return {
        'adapted_input': adapted_input,
        'patched_inputs': patched_inputs,
        'hidden': hidden,
        'decoder_hidden': hidden,
        'token_lengths': token_lengths,
        'logits': logits,
        'active_patch_size_bins': int(active_patch_size_bins),
        'active_patch_stride_bins': int(active_patch_stride_bins),
    }


def _evaluate_willett_gru(model, loader, device, blank_index, input_transform_config, active_bin_size_ms, patch_size_ms, patch_stride_ms, reference_patch_size_bins):
    model.eval()
    total_loss_sum = 0.0
    total_target_count = 0
    total_insertions = 0
    total_deletions = 0
    total_substitutions = 0
    total_reference_tokens = 0
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(device)
            input_lengths = batch['input_lengths'].to(device)
            labels = batch['labels'].to(device)
            label_lengths = batch['label_lengths'].to(device)
            x = prepare_timestep_flexible_inputs(
                x,
                input_lengths,
                config=input_transform_config,
                active_bin_size_ms=int(active_bin_size_ms),
                is_training=False,
            )
            outputs = _forward_willett_gru_resampled(
                model,
                x,
                input_lengths,
                batch['boundary_keys'],
                active_bin_size_ms=int(active_bin_size_ms),
                patch_size_ms=int(patch_size_ms),
                patch_stride_ms=int(patch_stride_ms),
                reference_patch_size_bins=int(reference_patch_size_bins),
            )
            loss_sum, target_count = compute_ctc_loss_sum(
                outputs['logits'],
                outputs['token_lengths'],
                labels,
                label_lengths,
                blank_index=int(blank_index),
            )
            if int(target_count) <= 0:
                continue
            total_loss_sum += float(loss_sum.item())
            total_target_count += int(target_count)
            predictions = ctc_greedy_decode(
                outputs['logits'],
                outputs['token_lengths'],
                blank_index=int(blank_index),
            )
            for row_idx, prediction in enumerate(predictions):
                reference = labels[row_idx, : int(label_lengths[row_idx].item())].tolist()
                insertions, deletions, substitutions = edit_counts(reference, prediction)
                total_insertions += int(insertions)
                total_deletions += int(deletions)
                total_substitutions += int(substitutions)
                total_reference_tokens += len(reference)
    if total_target_count <= 0 or total_reference_tokens <= 0:
        raise ValueError('Validation set produced zero effective targets during GRU evaluation.')
    total_errors = total_insertions + total_deletions + total_substitutions
    return {
        'val_ctc_bpphone': float(ctc_bits_per_target(total_loss_sum, total_target_count)),
        'val_phoneme_error_rate': float(total_errors / total_reference_tokens),
    }


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    payload = torch.load(GRU_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
except TypeError:
    payload = torch.load(GRU_CHECKPOINT_PATH, map_location='cpu')
checkpoint_cfg = dict(payload['config'])
problem = build_timestep_flexible_problem(
    cache_root=CACHE_ROOT,
    dataset=checkpoint_cfg.get('dataset', DATASET),
    feature_mode=checkpoint_cfg.get('feature_mode', FEATURE_MODE),
    boundary_key_mode=checkpoint_cfg.get('boundary_key_mode', 'session'),
    split_policy=checkpoint_cfg.get('split_policy', 'competition_train_test'),
    cv_num_folds=int(checkpoint_cfg.get('cv_num_folds', 5)),
    cv_fold_index=int(checkpoint_cfg.get('cv_fold_index', 0)),
)
sample_dim = _feature_dim_from_problem(problem, str(problem['feature_mode']))
train_adapter_keys = adapter_keys_from_rows(
    problem['train_rows'],
    dataset=str(problem['dataset']),
    boundary_key_mode=str(problem['boundary_key_mode']),
)
val_adapter_keys = adapter_keys_from_rows(
    problem['val_rows'],
    dataset=str(problem['dataset']),
    boundary_key_mode=str(problem['boundary_key_mode']),
)
session_adapter_keys = tuple(dict.fromkeys([*train_adapter_keys, *val_adapter_keys]))
model = WillettPhonemeModel(
    input_dim=int(sample_dim),
    vocab_size=int(problem['vocab']['num_classes']),
    patch_size=int(checkpoint_cfg.get('patch_size', 14)),
    patch_stride=int(checkpoint_cfg.get('patch_stride', 4)),
    input_projection_size=int(checkpoint_cfg.get('input_projection_size', 256)),
    input_projection_dropout=float(checkpoint_cfg.get('input_projection_dropout', 0.2)),
    decoder_backbone_type=str(checkpoint_cfg.get('decoder_backbone_type', 'gru')),
    gru_hidden_size=int(checkpoint_cfg.get('gru_hidden_size', 512)),
    gru_num_layers=int(checkpoint_cfg.get('gru_num_layers', 5)),
    gru_dropout=float(checkpoint_cfg.get('gru_dropout', 0.4)),
    s5_hidden_size=int(checkpoint_cfg.get('s5_hidden_size', 512)),
    s5_state_size=int(checkpoint_cfg.get('s5_state_size', 128)),
    s5_num_layers=int(checkpoint_cfg.get('s5_num_layers', 5)),
    s5_dropout=float(checkpoint_cfg.get('s5_dropout', 0.2)),
    s5_direction=str(checkpoint_cfg.get('s5_direction', 'causal')),
    s5_ffn_multiplier=float(checkpoint_cfg.get('s5_ffn_multiplier', 2.0)),
    s4d_hidden_size=int(checkpoint_cfg.get('s4d_hidden_size', 512)),
    s4d_state_size=int(checkpoint_cfg.get('s4d_state_size', 128)),
    s4d_num_layers=int(checkpoint_cfg.get('s4d_num_layers', 5)),
    s4d_dropout=float(checkpoint_cfg.get('s4d_dropout', 0.2)),
    s4d_direction=str(checkpoint_cfg.get('s4d_direction', 'causal')),
    s4d_ffn_multiplier=float(checkpoint_cfg.get('s4d_ffn_multiplier', 2.0)),
    session_adapter_keys=session_adapter_keys,
    session_adapter_enabled=bool(checkpoint_cfg.get('session_adapter_enabled', True)),
)
missing_keys, unexpected_keys = model.load_state_dict(payload['model_state'], strict=False)
if missing_keys or unexpected_keys:
    raise RuntimeError(
        'GRU checkpoint/model mismatch. '
        f'missing_keys={missing_keys} unexpected_keys={unexpected_keys}'
    )
model = model.to(device)
if str(checkpoint_cfg.get('decoder_backbone_type', 'gru')) != 'gru':
    raise ValueError('Expected a GRU checkpoint for this notebook baseline cell.')

smoothing_sigma_ms = float(checkpoint_cfg.get('input_smoothing_sigma_bins', 2.0)) * 20.0
smoothing_kernel_size_ms = float(checkpoint_cfg.get('input_smoothing_kernel_size', 100)) * 20.0
input_transform_config = TimestepFlexibleInputTransformConfig(
    input_smoothing_sigma_ms=smoothing_sigma_ms,
    input_smoothing_kernel_size_ms=smoothing_kernel_size_ms,
    input_smoothing_threshold=float(checkpoint_cfg.get('input_smoothing_threshold', 0.01)),
    white_noise_sd=0.0,
    constant_offset_sd=0.0,
)
reference_patch_size_bins = int(checkpoint_cfg.get('patch_size', 14))
patch_size_ms = int(PATCH_SIZE_MS)
patch_stride_ms = int(PATCH_STRIDE_MS)
blank_index = int(problem['vocab']['blank_index'])

results = []
for active_bin_size_ms in (20, 40):
    stats = _load_stats_for_bin(problem, checkpoint_cfg, sample_dim, active_bin_size_ms)
    missing_rows = normalization_stats_missing_rows(stats, problem['val_rows'])
    if missing_rows:
        preview = ', '.join(missing_rows[:5])
        raise ValueError(
            f'Normalization stats for {active_bin_size_ms} ms bins do not cover the validation rows. '
            f'First missing examples: {preview}'
        )
    dataset = RebinnedSequenceDataset(
        problem['val_rows'],
        cache_root=Path(problem['cache_root']),
        stats=stats,
        feature_mode=str(problem['feature_mode']),
        boundary_key_mode=str(problem['boundary_key_mode']),
        dataset=str(problem['dataset']),
        active_bin_size_ms=int(active_bin_size_ms),
    )
    loader = DataLoader(
        dataset,
        batch_sampler=make_length_aware_batch_sampler(
            problem['val_rows'],
            batch_size=int(checkpoint_cfg.get('batch_size', BATCH_SIZE)),
            shuffle=False,
            seed=int(checkpoint_cfg.get('seed', 7)) + 1,
            bin_size_ms=int(active_bin_size_ms),
        ),
        **loader_kwargs(device),
    )
    metrics = _evaluate_willett_gru(
        model,
        loader,
        device,
        blank_index,
        input_transform_config,
        active_bin_size_ms=int(active_bin_size_ms),
        patch_size_ms=int(patch_size_ms),
        patch_stride_ms=int(patch_stride_ms),
        reference_patch_size_bins=int(reference_patch_size_bins),
    )
    results.append({
        'model': 'Willett GRU',
        'bin_size_ms': int(active_bin_size_ms),
        **metrics,
    })

if S5_SUMMARY_PATH.exists():
    s5_summary = json.loads(S5_SUMMARY_PATH.read_text())
    for active_bin_size_ms in (20, 40):
        results.append({
            'model': 'Timestep-flexible S5',
            'bin_size_ms': int(active_bin_size_ms),
            'val_ctc_bpphone': _extract_s5_metric(s5_summary, active_bin_size_ms, 'ctc_bpphone'),
            'val_phoneme_error_rate': _extract_s5_metric(s5_summary, active_bin_size_ms, 'phoneme_error_rate'),
        })

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(['model', 'bin_size_ms']).reset_index(drop=True)
with pd.option_context('display.precision', 4):
    display(comparison_df[['model', 'bin_size_ms', 'val_ctc_bpphone', 'val_phoneme_error_rate']])

print('Saved Willett GRU best 20 ms reference from progress log:')
print(
    json.dumps(
        {
            'val_ctc_bpphone': GRU_SAVED_BEST_20MS_CTC,
            'val_phoneme_error_rate': GRU_SAVED_BEST_20MS_PER,
        },
        indent=2,
    )
)


## Experiment Helpers

Shared helpers for loading experiment summaries, rendering compact tables, and optionally appending notebook results back into the local `tests_and_results.md` note.


In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import display

RESULTS_NOTE_PATH = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments' / 'timestep_flexible_ssm' / 'tests_and_results.md'
WRITE_RESULTS_NOTE = False


def load_summary(path_like):
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(f'Missing summary file: {path}')
    return json.loads(path.read_text())


def append_results_note(title, lines):
    if not WRITE_RESULTS_NOTE:
        print(f'Skipping note append for {title!r}; set WRITE_RESULTS_NOTE=True to enable.')
        return
    note = RESULTS_NOTE_PATH.read_text() if RESULTS_NOTE_PATH.exists() else ''
    block = ['\n', f'## {title}', ''] + [str(line) for line in lines] + ['']
    RESULTS_NOTE_PATH.write_text(note.rstrip() + '\n' + '\n'.join(block))
    print(f'Appended results to: {RESULTS_NOTE_PATH}')


def compact_metric_table(rows):
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    cols = [col for col in ['model', 'train_condition', 'eval_condition', 'bin_size_ms', 'horizon_ms', 'val_ctc_bpphone', 'val_phoneme_error_rate', 'infonce_loss'] if col in frame.columns]
    return frame[cols]


## Experiment 1: Mixed-Bin Supervised Training

Balanced supervised training on `20 ms` and `40 ms` views. The S5 consumes the active bin size directly, while the GRU consumes duplicated dense `20 ms` frames for the `40 ms` half of the mixture.


In [ ]:
from timestep_flexible_ssm.supervised_experiments import (
    SupervisedExperimentConfig,
    run_mixed_bin_gru,
    run_mixed_bin_s5,
)

MIXED_OUTPUT_ROOT = OUTPUT_ROOT / 'mixed_bin_supervised'
MIXED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MIXED_S5_RUN_NAME = 'mixed_bin_s5_tx_only_colab'
MIXED_GRU_RUN_NAME = 'mixed_bin_gru_tx_only_colab'
RUN_MIXED_EXPERIMENT = False

mixed_config = SupervisedExperimentConfig(
    cache_root=CACHE_ROOT,
    output_root=MIXED_OUTPUT_ROOT,
    feature_mode=FEATURE_MODE,
    dataset=DATASET,
    normalization_mode=NORMALIZATION_MODE,
    batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    min_learning_rate=MIN_LEARNING_RATE,
    adam_epsilon=ADAM_EPSILON,
    patch_size_ms=PATCH_SIZE_MS,
    patch_stride_ms=PATCH_STRIDE_MS,
    session_adapter_enabled=SESSION_ADAPTER,
    input_smoothing_sigma_ms=40.0,
    input_smoothing_kernel_size_ms=2000.0,
    input_smoothing_threshold=0.01,
    mixed_bin_sizes_ms=(20, 40),
)

if RUN_MIXED_EXPERIMENT:
    mixed_s5_summary = run_mixed_bin_s5(
        SupervisedExperimentConfig(**{**asdict(mixed_config), 'run_name': MIXED_S5_RUN_NAME})
    )
    mixed_gru_summary = run_mixed_bin_gru(
        SupervisedExperimentConfig(
            **{
                **asdict(mixed_config),
                'run_name': MIXED_GRU_RUN_NAME,
                'learning_rate': 1e-2,
                'min_learning_rate': 1e-4,
                'adam_epsilon': 1e-1,
            }
        )
    )
    print(json.dumps({'mixed_s5': mixed_s5_summary['summary_path'], 'mixed_gru': mixed_gru_summary['summary_path']}, indent=2))
else:
    print('Set RUN_MIXED_EXPERIMENT=True to launch the mixed-bin training runs.')


In [ ]:
mixed_s5_summary_path = MIXED_OUTPUT_ROOT / MIXED_S5_RUN_NAME / 'summary.json'
mixed_gru_summary_path = MIXED_OUTPUT_ROOT / MIXED_GRU_RUN_NAME / 'summary.json'

rows = []
if summary_path.exists():
    baseline_s5 = load_summary(summary_path)
    rows.extend([
        {
            'model': 'Baseline S5',
            'train_condition': '20 ms only',
            'eval_condition': '20 ms',
            'bin_size_ms': 20,
            'val_ctc_bpphone': baseline_s5['best_metrics']['val_20ms_ctc_bpphone'],
            'val_phoneme_error_rate': baseline_s5['best_metrics']['val_20ms_phoneme_error_rate'],
        },
        {
            'model': 'Baseline S5',
            'train_condition': '20 ms only',
            'eval_condition': '40 ms',
            'bin_size_ms': 40,
            'val_ctc_bpphone': baseline_s5['best_metrics']['val_40ms_ctc_bpphone'],
            'val_phoneme_error_rate': baseline_s5['best_metrics']['val_40ms_phoneme_error_rate'],
        },
    ])
if mixed_s5_summary_path.exists():
    mixed_s5 = load_summary(mixed_s5_summary_path)
    rows.extend([
        {
            'model': 'Mixed S5',
            'train_condition': '50/50 20 ms + 40 ms',
            'eval_condition': '20 ms',
            'bin_size_ms': 20,
            'val_ctc_bpphone': mixed_s5['best_metrics']['val_20ms_ctc_bpphone'],
            'val_phoneme_error_rate': mixed_s5['best_metrics']['val_20ms_phoneme_error_rate'],
        },
        {
            'model': 'Mixed S5',
            'train_condition': '50/50 20 ms + 40 ms',
            'eval_condition': '40 ms',
            'bin_size_ms': 40,
            'val_ctc_bpphone': mixed_s5['best_metrics']['val_40ms_ctc_bpphone'],
            'val_phoneme_error_rate': mixed_s5['best_metrics']['val_40ms_phoneme_error_rate'],
        },
    ])
if mixed_gru_summary_path.exists():
    mixed_gru = load_summary(mixed_gru_summary_path)
    rows.extend([
        {
            'model': 'Mixed GRU',
            'train_condition': '50/50 20 ms + 40 ms duplicated',
            'eval_condition': '20 ms',
            'bin_size_ms': 20,
            'val_ctc_bpphone': mixed_gru['best_metrics']['val_20ms_ctc_bpphone'],
            'val_phoneme_error_rate': mixed_gru['best_metrics']['val_20ms_phoneme_error_rate'],
        },
        {
            'model': 'Mixed GRU',
            'train_condition': '50/50 20 ms + 40 ms duplicated',
            'eval_condition': '40 ms',
            'bin_size_ms': 40,
            'val_ctc_bpphone': mixed_gru['best_metrics']['val_40ms_ctc_bpphone'],
            'val_phoneme_error_rate': mixed_gru['best_metrics']['val_40ms_phoneme_error_rate'],
        },
    ])
frame = compact_metric_table(rows)
with pd.option_context('display.precision', 4):
    display(frame)
append_results_note('Mixed-Bin Supervised Results', frame.to_markdown(index=False).splitlines())


## Experiment 2: Missing-Bin Supervised Training

Random Bernoulli dropping with `p = 0.25`. The S5 consumes the irregular sequence with timestep deltas, while the GRU trains on interpolated dense inputs and evaluates with carry-forward reconstruction.


In [ ]:
from timestep_flexible_ssm.supervised_experiments import (
    SupervisedExperimentConfig,
    run_missing_bin_gru,
    run_missing_bin_s5,
)

MISSING_OUTPUT_ROOT = OUTPUT_ROOT / 'missing_bin_supervised'
MISSING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MISSING_S5_RUN_NAME = 'missing_bin_s5_tx_only_colab'
MISSING_GRU_RUN_NAME = 'missing_bin_gru_tx_only_colab'
RUN_MISSING_EXPERIMENT = False

missing_config = SupervisedExperimentConfig(
    cache_root=CACHE_ROOT,
    output_root=MISSING_OUTPUT_ROOT,
    feature_mode=FEATURE_MODE,
    dataset=DATASET,
    normalization_mode=NORMALIZATION_MODE,
    batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    min_learning_rate=MIN_LEARNING_RATE,
    adam_epsilon=ADAM_EPSILON,
    patch_size_ms=PATCH_SIZE_MS,
    patch_stride_ms=PATCH_STRIDE_MS,
    session_adapter_enabled=SESSION_ADAPTER,
    input_smoothing_sigma_ms=40.0,
    input_smoothing_kernel_size_ms=2000.0,
    input_smoothing_threshold=0.01,
    missing_drop_probability=0.25,
    train_mask_seed=101,
    val_mask_seed=202,
)

if RUN_MISSING_EXPERIMENT:
    missing_s5_summary = run_missing_bin_s5(
        SupervisedExperimentConfig(**{**asdict(missing_config), 'run_name': MISSING_S5_RUN_NAME})
    )
    missing_gru_summary = run_missing_bin_gru(
        SupervisedExperimentConfig(
            **{
                **asdict(missing_config),
                'run_name': MISSING_GRU_RUN_NAME,
                'learning_rate': 1e-2,
                'min_learning_rate': 1e-4,
                'adam_epsilon': 1e-1,
            }
        )
    )
    print(json.dumps({'missing_s5': missing_s5_summary['summary_path'], 'missing_gru': missing_gru_summary['summary_path']}, indent=2))
else:
    print('Set RUN_MISSING_EXPERIMENT=True to launch the missing-bin training runs.')


In [ ]:
missing_s5_summary_path = MISSING_OUTPUT_ROOT / MISSING_S5_RUN_NAME / 'summary.json'
missing_gru_summary_path = MISSING_OUTPUT_ROOT / MISSING_GRU_RUN_NAME / 'summary.json'

rows = []
if missing_s5_summary_path.exists():
    missing_s5 = load_summary(missing_s5_summary_path)
    rows.append({
        'model': 'Missing-bin S5',
        'train_condition': 'Bernoulli drop p=0.25',
        'eval_condition': 'Bernoulli drop p=0.25',
        'val_ctc_bpphone': missing_s5['best_metrics']['val_ctc_bpphone'],
        'val_phoneme_error_rate': missing_s5['best_metrics']['val_phoneme_error_rate'],
    })
if missing_gru_summary_path.exists():
    missing_gru = load_summary(missing_gru_summary_path)
    rows.append({
        'model': 'Missing-bin GRU',
        'train_condition': 'Interpolation train, carry-forward eval',
        'eval_condition': 'Bernoulli drop p=0.25',
        'val_ctc_bpphone': missing_gru['best_metrics']['val_ctc_bpphone'],
        'val_phoneme_error_rate': missing_gru['best_metrics']['val_phoneme_error_rate'],
    })
frame = compact_metric_table(rows)
with pd.option_context('display.precision', 4):
    display(frame)
append_results_note('Missing-Bin Supervised Results', frame.to_markdown(index=False).splitlines())


## Experiment 3: Future-Bin InfoNCE Runs

Separate future-prediction runs from canonical `20 ms` data, using normalized future frame bins as the positive targets at `20, 40, 60, 80, 100 ms` horizons.


In [ ]:
from timestep_flexible_ssm.future_infonce import FutureInfoNCEConfig, run_future_infonce

FUTURE_OUTPUT_ROOT = OUTPUT_ROOT / 'future_infonce'
FUTURE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FUTURE_S5_RUN_NAME = 'future_infonce_s5_tx_only_colab'
FUTURE_GRU_RUN_NAME = 'future_infonce_gru_tx_only_colab'
RUN_FUTURE_EXPERIMENT = False

future_config = FutureInfoNCEConfig(
    cache_root=CACHE_ROOT,
    output_root=FUTURE_OUTPUT_ROOT,
    feature_mode=FEATURE_MODE,
    dataset=DATASET,
    normalization_mode=NORMALIZATION_MODE,
    batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    min_learning_rate=MIN_LEARNING_RATE,
    adam_epsilon=ADAM_EPSILON,
    patch_size_ms=PATCH_SIZE_MS,
    patch_stride_ms=PATCH_STRIDE_MS,
    session_adapter_enabled=SESSION_ADAPTER,
    input_smoothing_sigma_ms=40.0,
    input_smoothing_kernel_size_ms=2000.0,
    input_smoothing_threshold=0.01,
    horizons_ms=(20, 40, 60, 80, 100),
    projection_dim=128,
)

if RUN_FUTURE_EXPERIMENT:
    future_s5_summary = run_future_infonce(
        FutureInfoNCEConfig(**{**asdict(future_config), 'run_name': FUTURE_S5_RUN_NAME, 'model_family': 's5'})
    )
    future_gru_summary = run_future_infonce(
        FutureInfoNCEConfig(
            **{
                **asdict(future_config),
                'run_name': FUTURE_GRU_RUN_NAME,
                'model_family': 'gru',
                'learning_rate': 1e-2,
                'min_learning_rate': 1e-4,
                'adam_epsilon': 1e-1,
            }
        )
    )
    print(json.dumps({'future_s5': future_s5_summary['summary_path'], 'future_gru': future_gru_summary['summary_path']}, indent=2))
else:
    print('Set RUN_FUTURE_EXPERIMENT=True to launch the future InfoNCE runs.')


In [ ]:
future_s5_summary_path = FUTURE_OUTPUT_ROOT / FUTURE_S5_RUN_NAME / 'summary.json'
future_gru_summary_path = FUTURE_OUTPUT_ROOT / FUTURE_GRU_RUN_NAME / 'summary.json'

frames = []
for model_name, path in [('Future S5', future_s5_summary_path), ('Future GRU', future_gru_summary_path)]:
    if not path.exists():
        continue
    summary = load_summary(path)
    metrics = dict(summary.get('metrics') or {})
    for key, value in metrics.items():
        if key.startswith('h') and key.endswith('_infonce_loss'):
            horizon_ms = int(key[1:].split('_', 1)[0])
            frames.append({
                'model': model_name,
                'horizon_ms': horizon_ms,
                'infonce_loss': float(value),
            })
frame = compact_metric_table(frames).sort_values(['model', 'horizon_ms']) if frames else pd.DataFrame()
with pd.option_context('display.precision', 4):
    display(frame)
append_results_note('Future InfoNCE Results', frame.to_markdown(index=False).splitlines() if not frame.empty else ['No future summaries yet.'])


## Final Comparison

Aggregate any completed experiment summaries into one notebook view. This is intentionally lightweight so you can rerun it after any subset of experiments completes.


In [ ]:
summary_rows = []

candidate_paths = [
    ('Baseline S5', summary_path),
    ('Mixed S5', MIXED_OUTPUT_ROOT / MIXED_S5_RUN_NAME / 'summary.json'),
    ('Mixed GRU', MIXED_OUTPUT_ROOT / MIXED_GRU_RUN_NAME / 'summary.json'),
    ('Missing-bin S5', MISSING_OUTPUT_ROOT / MISSING_S5_RUN_NAME / 'summary.json'),
    ('Missing-bin GRU', MISSING_OUTPUT_ROOT / MISSING_GRU_RUN_NAME / 'summary.json'),
    ('Future S5', FUTURE_OUTPUT_ROOT / FUTURE_S5_RUN_NAME / 'summary.json'),
    ('Future GRU', FUTURE_OUTPUT_ROOT / FUTURE_GRU_RUN_NAME / 'summary.json'),
]

for label, path in candidate_paths:
    if not Path(path).exists():
        continue
    summary = load_summary(path)
    metrics = dict(summary.get('best_metrics') or summary.get('metrics') or {})
    row = {
        'model': label,
        'artifact_path': str(path),
    }
    for key in ['val_20ms_ctc_bpphone', 'val_40ms_ctc_bpphone', 'val_20ms_phoneme_error_rate', 'val_40ms_phoneme_error_rate', 'mean_infonce_loss']:
        if key in metrics:
            row[key] = metrics[key]
    summary_rows.append(row)

summary_frame = pd.DataFrame(summary_rows)
with pd.option_context('display.precision', 4):
    display(summary_frame)
print('Results note path:', RESULTS_NOTE_PATH)
